# CNN 彩色图片分类：CIFAR-10 与 ResNet18

**功能**：学习分类 RGB 彩色图片，识别飞机、汽车、鸟、猫、鹿、狗、青蛙、马、船、卡车；包括训练、评估、保存和独立加载预测。

**方法**：torchvision CIFAR-10 → 32×32 RGB 图像 → 调整入口的 ResNet18 → 交叉熵与 Adam。网络结构独立一个代码单元格；固定训练 30 轮，保存最后一轮权重。

**注意事项**：本文件未执行。首次运行需联网下载数据，默认完整划分为训练 40,000／监控 10,000／官方测试 10,000 张。强制 GPU，优先 CUDA，其次 Apple MPS；均不可用时直接报错。图像、模型及 one-hot 训练标签统一为 float32。全量 ResNet18 比小规模手写数字示例耗时更长；每轮还会评估完整训练集和监控集。运行后生成当前目录下的 data/ 与 ColorPicture_model.pt。

**依赖**：Python、PyTorch、torchvision、NumPy、Matplotlib。

**参考**：`nndl-practice/pytorch/chap5卷积神经网络/卷积神经网络-下.ipynb` 的 CIFAR-10／ResNet18 和 Adam 配置。训练循环沿用前述本地 MNIST 参考并按 machine-learning 技能区分 train/eval。文件自包含，不依赖参考仓库的 nndl 模块。


## 1. 导入库与学习配置

默认使用官方 50,000 张训练图片，按 80%/20% 随机划分为训练和测试监控集；官方 10,000 张测试图片训练结束后才评估。两个样本数参数为 `None` 时使用完整数据。

如需先熟悉流程，可在运行前把 TRAIN_POOL_SIZE 改为 6250、FINAL_TEST_SIZE 改为 1000，得到 5,000／1,250／1,000 张。小样本结果不能代表完整基准。轮数和学习率提前固定，监控及最终测试结果不用于选择权重或调参；调参应另设验证集。

类别顺序：airplane、automobile、bird、cat、deer、dog、frog、horse、ship、truck。


In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 100
TRAIN_POOL_SIZE = None  # None：使用官方训练集全部 50,000 张
FINAL_TEST_SIZE = None   # None：使用官方测试集全部 10,000 张
TRAIN_FRACTION = 0.80
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.005
DTYPE = torch.float32
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
DATA_DIR = Path.cwd() / "data"
MODEL_PATH = Path.cwd() / "ColorPicture_model.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# 强制使用 GPU；MPS 不支持 float64，因此模型和图像统一使用 float32。
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.mps.manual_seed(SEED)
else:
    raise RuntimeError("必须使用 GPU：未检测到可用的 CUDA 或 Apple MPS。")
print("device:", device, "dtype:", DTYPE)


## 2. RGB 预处理与数据划分

CIFAR-10 原图为 32×32 RGB 彩色图。ToTensor 把像素映射到 [0,1]，随后三个通道分别按 `(x−0.5)/0.5` 映射到 [−1,1]。这是固定变换，不从监控或测试集拟合统计量。

批量图像形状为 `[B,3,32,32]`，one-hot 标签为 `[B,10]`，均为 float32。本入门示例暂不使用随机裁剪或翻转，以便理解基本数据流。


In [ ]:
preprocess_config = {"image_size": 32, "mean": (0.5, 0.5, 0.5), "std": (0.5, 0.5, 0.5)}
transform = transforms.Compose([
    transforms.Resize((preprocess_config["image_size"],) * 2),
    transforms.ToTensor(),
    transforms.Normalize(preprocess_config["mean"], preprocess_config["std"]),
])
# 将类别编号转换为 float32 one-hot 标签，不构造 int64 训练目标。
# 例如类别 3（cat） → [0,0,0,1,0,0,0,0,0,0]。
def encode_label(label):
    return torch.tensor([float(label == digit) for digit in range(10)], dtype=DTYPE)

cifar_train = datasets.CIFAR10(DATA_DIR, train=True, download=True,
                             transform=transform, target_transform=encode_label)
cifar_test = datasets.CIFAR10(DATA_DIR, train=False, download=True,
                            transform=transform, target_transform=encode_label)

assert cifar_train.classes == class_names
assert cifar_test.classes == class_names
rng = np.random.default_rng(SEED)
train_count = len(cifar_train) if TRAIN_POOL_SIZE is None else TRAIN_POOL_SIZE
final_count = len(cifar_test) if FINAL_TEST_SIZE is None else FINAL_TEST_SIZE
if not 2 <= train_count <= len(cifar_train):
    raise ValueError("TRAIN_POOL_SIZE 必须在 2 与官方训练集大小之间")
if not 1 <= final_count <= len(cifar_test):
    raise ValueError("FINAL_TEST_SIZE 超出官方测试集范围")
indices = rng.permutation(len(cifar_train))[:train_count]
split = int(TRAIN_FRACTION * train_count)
if not 0 < split < train_count:
    raise ValueError("训练集和监控集都必须非空")
train_indices, monitor_indices = indices[:split], indices[split:]
final_indices = rng.permutation(len(cifar_test))[:final_count]
assert np.intersect1d(train_indices, monitor_indices).size == 0

train_dataset = Subset(cifar_train, train_indices.tolist())
monitor_dataset = Subset(cifar_train, monitor_indices.tolist())
final_dataset = Subset(cifar_test, final_indices.tolist())
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), num_workers=0)
# 固定顺序的训练集评估 loader，避免改变训练 loader 的随机批次顺序。
train_eval_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
monitor_loader = DataLoader(monitor_dataset, batch_size=BATCH_SIZE, shuffle=False)
final_loader = DataLoader(final_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("训练 / 监控 / 官方测试:", len(train_dataset), len(monitor_dataset), len(final_dataset))
images, labels = next(iter(train_eval_loader))
assert images.shape[1:] == (3, 32, 32)
assert torch.isfinite(images).all()
assert labels.shape == (len(images), 10)
assert labels.dtype == DTYPE
assert ((labels == 0) | (labels == 1)).all()
assert (labels.sum(dim=1) == 1).all()
print("图像批量形状:", tuple(images.shape), "标签形状:", tuple(labels.shape))


## 3. 查看训练图片

显示前将 [−1,1] 还原到 [0,1]，将 `[C,H,W]` 转换为 Matplotlib 的 `[H,W,C]`。标题显示类别名称。


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        image = (images[i] * 0.5 + 0.5).clamp(0, 1)
        ax.imshow(image.permute(1, 2, 0).numpy())
        ax.set_title(class_names[labels[i].argmax().item()])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. 网络结构（独立代码单元格）

采用参考的 ResNet18，weights=None 表示从零训练。将首层改为三通道 3×3、步长 1 卷积，移除入口最大池化，保留小图片的空间细节。

| 模块 | 单个样本输出形状 |
|---|---|
| RGB 输入 | 3×32×32 |
| conv1、BatchNorm、ReLU | 64×32×32 |
| layer1：2 个残差块 | 64×32×32 |
| layer2：2 个残差块 | 128×16×16 |
| layer3：2 个残差块 | 256×8×8 |
| layer4：2 个残差块 | 512×4×4 |
| 全局平均池化与展平 | 512 |
| 全连接 | 10 |

残差块将输入经 shortcut 后加到卷积分支；通道或空间形状变化时使用投影分支对齐。输出是十类 logits，训练时不另加 softmax。下一单元打印完整网络，可观察残差块的具体层。


In [ ]:
# 网络结构：本单元格仅定义用于 32×32 RGB 图像的 ResNet18。
from torchvision.models import resnet18

torch.manual_seed(SEED)
model = resnet18(weights=None, num_classes=10)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()


## 5. 设备、损失函数与优化器

模型、图像和 one-hot 训练标签均为 float32，放到同一 GPU。交叉熵直接接收 one-hot 概率目标。argmax 返回的整数类别索引只用于计数和显示，不用作训练标签。

沿用参考的 Adam，学习率 0.001，weight_decay=0.005；这里是 Adam 的 L2 惩罚，与 AdamW 解耦权重衰减不同。


In [ ]:
model = model.to(device=device, dtype=DTYPE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(model)
print("可训练参数:", sum(p.numel() for p in model.parameters() if p.requires_grad))
model.eval()
with torch.no_grad():
    sample_logits = model(images.to(device=device, dtype=DTYPE))
assert sample_logits.shape == (len(images), 10)
print("输出形状:", tuple(sample_logits.shape))


## 6. 训练与评估函数

训练使用 `model.train()`：清空梯度 → 前向传播 → 交叉熵 → 反向传播 → 更新参数。评估使用 `model.eval()` 和 `torch.no_grad()`，不更新参数。

每轮训练结束后，在同一组固定权重下重新遍历整个训练集和监控集，按样本数加权计算平均损失；不把训练期间参数不断变化的批量损失与监控损失比较。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    for xb, yb in loader:
        xb = xb.to(device=device, dtype=DTYPE)
        yb = yb.to(device=device, dtype=DTYPE)
        if not torch.isfinite(xb).all():
            raise ValueError("输入图像包含 NaN/Inf")
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        if not torch.isfinite(loss):
            raise ValueError("训练损失出现 NaN/Inf")
        loss.backward()
        optimizer.step()


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device=device, dtype=DTYPE)
            yb = yb.to(device=device, dtype=DTYPE)
            logits = model(xb)
            loss = criterion(logits, yb)
            if not torch.isfinite(logits).all() or not torch.isfinite(loss):
                raise ValueError("评估输出或损失出现 NaN/Inf")
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb.argmax(dim=1)).sum().item()
            count += len(yb)
    return total_loss / count, correct / count


## 7. 固定轮数训练

每轮更新完成后，在 eval 模式下重新计算完整训练集与监控集的损失和准确率。BatchNorm 评估时使用已有统计量，不更新运行均值。

默认固定 30 轮，每轮全量训练约 625 次参数更新；不早停、不选择最低监控损失的权重。重新训练时从网络结构单元重建模型和优化器；重复完整实验时从配置单元重置随机种子。


In [ ]:
train_loss_history, test_loss_history = [], []
train_accuracy_history, test_accuracy_history = [], []
for epoch in range(1, EPOCHS + 1):
    train_one_epoch(model, train_loader, criterion, optimizer)
    train_loss, train_accuracy = evaluate(model, train_eval_loader, criterion)
    monitor_loss, monitor_accuracy = evaluate(model, monitor_loader, criterion)
    train_loss_history.append(train_loss)
    test_loss_history.append(monitor_loss)
    train_accuracy_history.append(train_accuracy)
    test_accuracy_history.append(monitor_accuracy)
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss={train_loss:.4f}, accuracy={train_accuracy:.2%} | "
          f"monitor loss={monitor_loss:.4f}, accuracy={monitor_accuracy:.2%}")
model.eval()


## 8. 损失与准确率曲线（独立单元格）

蓝色实线表示训练集，红色虚线表示测试监控集。损失使用对数坐标；准确率表示正确分类的比例。两条曲线明显分离可能提示过拟合，但不能仅凭一次实验作结论。

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, train_loss_history, color="blue", linestyle="-", label="Training")
axes[0].plot(epochs, test_loss_history, color="red", linestyle="--", label="Test monitoring")
axes[0].set_yscale("log")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[1].plot(epochs, train_accuracy_history, color="blue", linestyle="-", label="Training")
axes[1].plot(epochs, test_accuracy_history, color="red", linestyle="--", label="Test monitoring")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


## 9. 官方测试集的最终评估

此集合未参与训练或逐轮监控。模型配置冻结后评估一次；如果根据这里的结果调整模型，它就不再是独立的最终测试集。

In [ ]:
final_loss, final_accuracy = evaluate(model, final_loader, criterion)
print(f"官方测试样本数: {len(final_dataset)}")
print(f"Cross-entropy: {final_loss:.4f}")
print(f"Accuracy: {final_accuracy:.2%}")


## 10. 保存完整模型与预处理参数

保存最后一轮完整模型、固定预处理参数、配置和样本索引。预测时不用重建网络或重新训练。

In [ ]:
model.eval()
# 深拷贝到 CPU 保存，避免改变当前模型所在设备。
import copy
saved_model = copy.deepcopy(model).cpu()
torch.save({
    "model": saved_model,
    "preprocess": preprocess_config,
    "dtype": "float32",
    "class_names": class_names,
    "config": {"seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
               "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
               "train_fraction": TRAIN_FRACTION},
    "train_indices": train_indices.tolist(),
    "monitor_indices": monitor_indices.tolist(),
    "final_indices": final_indices.tolist(),
    "history": {"train_loss": train_loss_history, "monitor_loss": test_loss_history,
                "train_accuracy": train_accuracy_history,
                "monitor_accuracy": test_accuracy_history},
}, MODEL_PATH)
print("保存至:", MODEL_PATH)


## 11. 独立加载与预测

可在新内核中只运行以下加载与显示单元，无需运行训练部分。仅加载自己生成的可信完整模型（weights_only=False），复用保存的 RGB 预处理参数，强制 GPU 预测。

这里显示已完成最终评估的官方测试图片。softmax 最大值是概率分数，不保证已经校准。自己的图片也应使用 RGB 和同样的归一化；模型只认识 CIFAR-10 十类，其他内容也会被分到其中一类。


In [ ]:
from pathlib import Path
import torch
from torchvision import datasets, transforms

# 独立预测同样强制使用 GPU，不依赖训练单元格中的设备变量。
if torch.cuda.is_available():
    prediction_device = torch.device("cuda")
elif torch.backends.mps.is_available():
    prediction_device = torch.device("mps")
else:
    raise RuntimeError("预测必须使用 GPU：未检测到可用的 CUDA 或 Apple MPS。")
checkpoint_path = Path.cwd() / "ColorPicture_model.pt"
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
loaded_model = saved["model"].to(device=prediction_device, dtype=torch.float32)
loaded_model.eval()
p = saved["preprocess"]
prediction_class_names = saved["class_names"]
prediction_transform = transforms.Compose([
    transforms.Resize((p["image_size"],) * 2),
    transforms.ToTensor(),
    transforms.Normalize(p["mean"], p["std"]),
])
def encode_prediction_label(label):
    return torch.tensor([float(label == digit) for digit in range(10)], dtype=torch.float32)

prediction_dataset = datasets.CIFAR10(Path.cwd() / "data", train=False,
                                    download=True, transform=prediction_transform,
                                    target_transform=encode_prediction_label)
show_indices = saved["final_indices"][:10]
examples = [prediction_dataset[i] for i in show_indices]
prediction_images = torch.stack([image for image, label in examples])
true_targets = torch.stack([label for image, label in examples])
# argmax 返回类别索引，只用于显示/计数，不作为交叉熵的训练标签。
true_labels = true_targets.argmax(dim=1)
with torch.no_grad():
    logits = loaded_model(prediction_images.to(device=prediction_device, dtype=torch.float32))
    probabilities = torch.softmax(logits, dim=1).cpu()
    confidence, predictions = probabilities.max(dim=1)
for truth, pred, score in zip(true_labels.tolist(), predictions.tolist(), confidence.tolist()):
    print(f"真实类别: {prediction_class_names[truth]}, "
          f"预测类别: {prediction_class_names[pred]}, 概率分数: {score:.3f}")


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
mean = torch.tensor(p["mean"], dtype=torch.float32).view(3, 1, 1)
std = torch.tensor(p["std"], dtype=torch.float32).view(3, 1, 1)
for i, ax in enumerate(axes.flat):
    if i < len(prediction_images):
        image = (prediction_images[i] * std + mean).clamp(0, 1)
        ax.imshow(image.permute(1, 2, 0).numpy())
        truth = prediction_class_names[true_labels[i].item()]
        pred = prediction_class_names[predictions[i].item()]
        color = "black" if predictions[i] == true_labels[i] else "red"
        ax.set_title(f"True: {truth}\nPred: {pred}", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
